In [1]:
import pandas as pd
from preprocessing import preparar_todo_el_dataset
X_train_procesado, X_test_procesado, y_train, y_test, df_limpio, X_train, X_test, pipeline = preparar_todo_el_dataset()

print(f"Entrenaremos con {X_train_procesado.shape[0]} clientes y probaremos con {X_test_procesado.shape[0]} clientes\n")

Entrenaremos con 32686 clientes y probaremos con 8172 clientes



In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Usamos class_weight='balanced' para que el modelo le preste atención al 11% que SÍ compra
modelo_logistico = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# ENTRENAR EL MODELO (Aprender de los datos de entrenamiento)
modelo_logistico.fit(X_train_procesado, y_train)
print("Modelo de Regresión Logística entrenado")

# HACER PREDICCIONES Ponerlo a prueba con los datos que nunca ha visto, con x_test
predicciones_logistica = modelo_logistico.predict(X_test_procesado)

print("\n--- MÉTRICAS ---")
print(classification_report(y_test, predicciones_logistica))

print("\n--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, predicciones_logistica))

Modelo de Regresión Logística entrenado

--- MÉTRICAS ---
              precision    recall  f1-score   support

           0       0.95      0.82      0.88      7251
           1       0.31      0.66      0.42       921

    accuracy                           0.80      8172
   macro avg       0.63      0.74      0.65      8172
weighted avg       0.88      0.80      0.83      8172


--- MATRIZ DE CONFUSIÓN ---
[[5919 1332]
 [ 317  604]]


In [3]:
from sklearn.ensemble import RandomForestClassifier

# MODELO RANDOM FOREST
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

# ENTRENAR EL MODELO
modelo_rf.fit(X_train_procesado, y_train)

# PREDICCIONES
predicciones_rf = modelo_rf.predict(X_test_procesado)

# METRICAS
print("\n--- REPORTE DE MÉTRICAS: RANDOM FOREST ---")
print(classification_report(y_test, predicciones_rf))

print("\n--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, predicciones_rf))


--- REPORTE DE MÉTRICAS: RANDOM FOREST ---
              precision    recall  f1-score   support

           0       0.95      0.87      0.91      7251
           1       0.37      0.61      0.46       921

    accuracy                           0.84      8172
   macro avg       0.66      0.74      0.68      8172
weighted avg       0.88      0.84      0.86      8172


--- MATRIZ DE CONFUSIÓN ---
[[6306  945]
 [ 359  562]]


In [ ]:
# Nuevo Ramdon forest con clusteres del mkmeasn para mejorar

import joblib
from sklearn.preprocessing import OneHotEncoder

#CARGAR LOS OBJETOS SERIALIZADOS DE TU CLUSTERING

scaler_kmeans = joblib.load('models/scaler_numerico_v1.joblib')
modelo_kmeans = joblib.load('models/kmeans_numericas.joblib')

# GENERAR LOS CLÚSTERES PARA TRAIN Y TEST
# las variables numéricas que uso K-Means 
cols_numericas_kmeans = ['age', 'campaign', 'emp.var.rate', 'cons.price.idx', 'euribor3m', 'nr.employed']

# Escalamos y predecimos el clúster para el set de Entrenamiento
X_train_num_scaled = scaler_kmeans.transform(X_train[cols_numericas_kmeans])
clusters_train = modelo_kmeans.predict(X_train_num_scaled)

# Escalamos y predecimos el clúster para el set de Prueba
X_test_num_scaled = scaler_kmeans.transform(X_test[cols_numericas_kmeans])
clusters_test = modelo_kmeans.predict(X_test_num_scaled)

# TRANSFORMAR LOS CLÚSTERES A ONE-HOT ENCODING

encoder_cluster = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Ajustamos el encoder con los clústeres de train y transformamos ambos
clusters_train_ohe = encoder_cluster.fit_transform(clusters_train.reshape(-1, 1))
clusters_test_ohe = encoder_cluster.transform(clusters_test.reshape(-1, 1))

# AGREGAR LOS CLÚSTERES A LAS MATRICES PROCESADAS ORIGINALES

import numpy as np
X_train_con_clusters = np.hstack((X_train_procesado, clusters_train_ohe))
X_test_con_clusters = np.hstack((X_test_procesado, clusters_test_ohe))

print(f"Forma original de X_train_procesado: {X_train_procesado.shape}")
print(f"Nueva forma con columnas de clústeres: {X_train_con_clusters.shape}")



Forma original de X_train_procesado: (32686, 29)
Nueva forma con columnas de clústeres: (32686, 33)


In [7]:
print("\nEntrenando Random Forest con Clústeres...")
modelo_rf_clusters = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

modelo_rf_clusters.fit(X_train_con_clusters, y_train)


# EVALUAR EL NUEVO MODELO
predicciones_rf_clusters = modelo_rf_clusters.predict(X_test_con_clusters)

print("\n--- METRICAS: RANDOM FOREST + CLUSTERS ---")
print(classification_report(y_test, predicciones_rf_clusters))

print("\n--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, predicciones_rf_clusters))


Entrenando Random Forest con Clústeres...

--- METRICAS: RANDOM FOREST + CLUSTERS ---
              precision    recall  f1-score   support

           0       0.95      0.85      0.90      7251
           1       0.35      0.63      0.45       921

    accuracy                           0.83      8172
   macro avg       0.65      0.74      0.68      8172
weighted avg       0.88      0.83      0.85      8172


--- MATRIZ DE CONFUSIÓN ---
[[6195 1056]
 [ 343  578]]


In [ ]:
# NUevo modelo de regresion pero ahora usando clusteres calculados por kmeans

# CREAR EL MODELO
# Usamos los mismos parámetros que en nuestro Baseline para que la comparación sea justa
modelo_logistico_clusters = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# ENTRENAR CON LOS DATOS ENRIQUECIDOS
modelo_logistico_clusters.fit(X_train_con_clusters, y_train)

#  HACER PREDICCIONES
predicciones_log_clusters = modelo_logistico_clusters.predict(X_test_con_clusters)

# EVALUAR RESULTADOS
print("\n--- MÉTRICAS: REGRESIÓN LOGÍSTICA + CLUSTERS ---")
print(classification_report(y_test, predicciones_log_clusters))

print("\n--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, predicciones_log_clusters))


--- MÉTRICAS: REGRESIÓN LOGÍSTICA + CLUSTERS ---
              precision    recall  f1-score   support

           0       0.95      0.80      0.87      7251
           1       0.30      0.67      0.41       921

    accuracy                           0.78      8172
   macro avg       0.62      0.73      0.64      8172
weighted avg       0.88      0.78      0.82      8172


--- MATRIZ DE CONFUSIÓN ---
[[5798 1453]
 [ 305  616]]
